In [42]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
### setup ###

import sys
import pandas as pd
# setup path to python virtual env & code directory
ecf_env_path = '/Users/kszczepaniak/Code/ecf-explorer-env'
lib_ecf_path = ecf_env_path + '/ecf-explorer/src'
sys.path.append(lib_ecf_path)
from clustering.utils import split_dataset_df_into_single_proteins, create_bash_script_to_run_clustering
from clustering.utils import run_clustering_with_bash, concatenate_clustering_results
from clustering.utils import clean_clustering_partial_data
from utils.preprocess import setup_dir_tree, load_and_filter_data

workdir_path         = '/Users/kszczepaniak/Data/Phage/phage-pp-workdir-ecftest-refseq/ecf-clusters'
input_table_filepath = workdir_path + '/table-hhr.txt'
setup_dir_tree(input_table_filepath)

/ecf-explorer/ directory already set up
/ecf-explorer/input/ directory already set up
/ecf-explorer/input/annots/ directory already set up
/ecf-explorer/input/all-vs-all/ directory already set up
/ecf-explorer/input/annots/hhrepid// directory already set up
/ecf-explorer/intermediate/ directory already set up
/ecf-explorer/intermediate/protein_dfs/ directory already set up
/ecf-explorer/intermediate/qecf_msa/ directory already set up
/ecf-explorer/intermediate/qecf_dfs/ directory already set up
/ecf-explorer/intermediate/qecf_all_by_all/ directory already set up
/ecf-explorer/intermediate/network/ directory already set up
/ecf-explorer/output/ directory already set up
/ecf-explorer/plots/ directory already set up
/ecf-explorer/scripts/ directory already set up


In [45]:
### table of superECFs
mcl_out_dirpath = '/Users/kszczepaniak/Data/Phage/results-ECF-refseq-hhblits-102022/ECF-network/'

# create subtables ECF to superECF and joi them on ecf id
secf_tables = {}
for prob, cov in [(50, 30), (50, 50), (50, 80), (70, 30), (70, 50), (70, 80), (95, 30), (95, 50), (95, 80)]:
    mcl_out_fpath = mcl_out_dirpath + 'repr-hits-pairwise-prob{}-cov{}-mcl.out'.format(prob, cov)
    ecf2secf = {}
    with open(mcl_out_fpath, 'r') as fout:
        for n, line in enumerate(fout):
            for i, ecf in enumerate(line.strip().split('\t')):
                ecf2secf[str(n) + '_'+ str(i)] = [ecf, 'secf' + str(n)]
    secf_table = pd.DataFrame.from_dict(ecf2secf, orient='index', columns=['ecf', 'secf-p{}-c{}'.format(prob, cov)])
    secf_tables['table-p{}-c{}'.format(prob, cov)] = secf_table

In [51]:
table_50 = secf_tables['table-p50-c30'].join(secf_tables['table-p50-c50'].set_index('ecf'), on='ecf').join(secf_tables['table-p50-c80'].set_index('ecf'), on='ecf')
table_70 = secf_tables['table-p70-c30'].join(secf_tables['table-p70-c50'].set_index('ecf'), on='ecf').join(secf_tables['table-p70-c80'].set_index('ecf'), on='ecf')
table_95 = secf_tables['table-p95-c30'].join(secf_tables['table-p95-c50'].set_index('ecf'), on='ecf').join(secf_tables['table-p95-c80'].set_index('ecf'), on='ecf')
table_all = table_50.join(table_70.set_index('ecf'), on='ecf').join(table_95.set_index('ecf'), on='ecf')
table_all

,ecf,secf-p50-c30,secf-p50-c50,secf-p50-c80,secf-p70-c30,secf-p70-c50,secf-p70-c80,secf-p95-c30,secf-p95-c50,secf-p95-c80
0_0,ecf_cluster2973,secf0,secf0,secf4,secf0,secf0,secf4,secf0,secf0,secf4
0_1,ecf_cluster2655,secf0,secf0,secf5646,secf0,secf0,secf5650,secf0,secf0,secf5681
0_2,ecf_cluster7065,secf0,secf0,secf4,secf0,secf0,secf4,secf0,secf0,secf4
0_3,ecf_cluster4594,secf0,secf0,secf4,secf0,secf0,secf4,secf0,secf0,secf4
0_4,ecf_cluster1017,secf0,secf0,secf6,secf0,secf0,secf6,secf0,secf0,secf5
...,...,...,...,...,...,...,...,...,...,...
3103_0,ecf_cluster3407,secf3103,secf3865,secf2408,secf3169,secf3879,secf2410,secf3368,secf3950,secf2416
3104_0,ecf_cluster6555,secf3104,secf2989,secf6284,secf3170,secf2997,secf6288,secf3369,secf3044,secf6327
3105_0,ecf_cluster2668,secf3105,secf4061,secf1775,secf3171,secf4078,secf1776,secf3370,secf4158,secf1775
3106_0,ecf_cluster6392,secf3106,secf2947,secf2279,secf3172,secf2954,secf2281,secf3371,secf3000,secf2285


In [52]:
table_all.to_csv(mcl_out_dirpath + 'table-ecf-clustering.csv', index=False)